# Análise de Consumo de Energia - Notebook
Este notebook lê um arquivo CSV do S3 e realiza diversas manipulações com `pandas` conforme os requisitos.

In [107]:
pip install boto3

Note: you may need to restart the kernel to use updated packages.


In [108]:
import boto3
import pandas as pd
from io import StringIO

## Leitura do arquivo CSV do S3

In [109]:
session = boto3.Session(profile_name='user3')
s3 = session.client('s3')

bucket_name = 'isabellabucket123'
object_key = 'basedados.csv'

csv_obj = s3.get_object(Bucket=bucket_name, Key=object_key)
body = csv_obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(body), sep=';')

df.head()

,Cidade,Campus,Ano,Mês,Quantidade em kWh
0,Ituiutaba,Pontal,2023,Janeiro,52150
1,Ituiutaba,Pontal,2023,Fevereiro,44800
2,Ituiutaba,Pontal,2023,Março,62300
3,Ituiutaba,Pontal,2023,Abril,56000
4,Ituiutaba,Pontal,2023,Maio,54250


## Filtro: Uberlândia e Julho

In [110]:
# Remover espaços extras nas colunas antes de aplicar o filtro
df['Cidade'] = df['Cidade'].str.strip()
df['Mês'] = df['Mês'].str.strip()

# Filtro 
filtro = (df['Cidade'] == 'Uberlândia') & (df['Mês'] == 'Julho')
df_filtrado = df[filtro]

# Visualizar 
display(df_filtrado)
df_filtrado.to_csv('filtro.csv', index=False)


,Cidade,Campus,Ano,Mês,Quantidade em kWh
30,Uberlândia,Educaço Física,2023,Julho,41370
42,Uberlândia,Fazendas,2023,Julho,49455
54,Uberlândia,Glória,2023,Julho,25200
66,Uberlândia,Santa Mônica,2023,Julho,295400
78,Uberlândia,Umuarama,2023,Julho,200200


## Agregação: Consumo total por campus

In [111]:
# Agrupar por Campus e Ano, somando o consumo
agregado = df.groupby(['Campus', 'Ano'])['Quantidade em kWh'].sum().reset_index()

# Visualizar os dados agregados
display(agregado)

# Exportar para CSV
agregado.to_csv('consumo_total_por_ano_campus.csv', index=False)


,Campus,Ano,Quantidade em kWh
0,Araras,2023,497560
1,Educaço Física,2023,521430
2,Fazendas,2023,650489
3,Glória,2023,393050
4,Pontal,2023,681800
5,Santa Mônica,2023,4754400
6,Umuarama,2023,3286200


## Condicional: consumo alto (> 100.000 kWh)

In [112]:
df['Consumo Alto'] = df['Quantidade em kWh'].apply(lambda x: 'Sim' if x > 100000 else 'Não')
display(df.head())
df.to_csv('consumo_alto.csv', index=False)


,Cidade,Campus,Ano,Mês,Quantidade em kWh,Consumo Alto
0,Ituiutaba,Pontal,2023,Janeiro,52150,Não
1,Ituiutaba,Pontal,2023,Fevereiro,44800,Não
2,Ituiutaba,Pontal,2023,Março,62300,Não
3,Ituiutaba,Pontal,2023,Abril,56000,Não
4,Ituiutaba,Pontal,2023,Maio,54250,Não
